<a href="https://colab.research.google.com/github/oleg61/--AI--/blob/main/3_2_%D0%A0%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D0%B5_%D0%92%D0%BE%D1%80%D0%BE%D0%BF%D0%B0%D0%B5%D0%B2_%D0%9E%D0%A1_%D1%80%D0%B5%D1%88%D0%B5%D0%BD%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain openai langchain-openai langchain-community -q -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 986.1 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import re
import pandas as pd
from tqdm import tqdm
from getpass import getpass

from langchain.prompts import PromptTemplate
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
from langchain_openai import ChatOpenAI  # или другой LLM по вашему выбору

from langchain_openai import ChatOpenAI
from getpass import getpass

#course_api_key= "Введите ваш API ключ, полученный в боте курса"
course_api_key = getpass(prompt='Введите ваш API ключ, полученный в боте курса')

# инициализируем языковую модель
llm = ChatOpenAI(api_key=course_api_key, model='gpt-4o-mini',
                 base_url="https://aleron-llm.neuraldeep.tech/")






Введите ваш API ключ, полученный в боте курса··········


In [ ]:
# Загрузка данных
df = pd.read_csv("https://stepik.org/media/attachments/lesson/1110883/raw_texts.csv")
df.head()



,raw_text
0,"The sun was setting, casting long shadows over..."
1,"Le soleil se couchait, jetant de longues ombre..."
2,"El sol se estaba poniendo, proyectando largas ..."
3,"La ciudad estaba llena de vida, sus calles lle..."
4,"La ville était pleine de vie, ses rues remplie..."


In [ ]:
# Функция очистки текста
def clean_text(inputs: dict) -> dict:
    text = inputs["text"]
    # Удаляем нежелательные символы
    for char in ["¿", "¡", "£"]:
        text = text.replace(char, "")
    return {"text": text}

# Определяем схемы ответа
language_schema = ResponseSchema(
    name="language",
    description="The language of the text, in English (e.g., 'German', 'French', 'English')."
)

person_schema = ResponseSchema(
    name="main_character",
    description="The name of the main character in the text, written in the original language of the text."
)

response_schemas = [language_schema, person_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

# Шаблон промпта
prompt = PromptTemplate(
    template="Analyze the following literary excerpt:\n\n{text}\n\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": format_instructions}
)

# Инициализируем LLM (предполагается, что у вас установлен OPENAI_API_KEY)
#llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Создаём цепочку с помощью LCEL
chain = prompt | llm | output_parser

# Применяем цепочку к каждому тексту
results = []
for raw_text in tqdm(df['raw_text']):
    cleaned = clean_text({"text": raw_text})
    try:
        response = chain.invoke({"text": cleaned["text"]})
        results.append({
            "text": cleaned["text"],
            "language": response.get("language", "").strip(),
            "main_character": response.get("main_character", "").strip()
        })
    except Exception as e:
        # В случае ошибки — заполняем пустыми значениями
        results.append({
            "text": cleaned["text"],
            "language": "",
            "main_character": ""
        })

# Создаём итоговый DataFrame
result_df = pd.DataFrame(results)

# Сохраняем в файл
result_df[['text', 'language', 'main_character']].to_csv('3.2.9_solution.csv', index=False)

100%|██████████| 13/13 [00:22<00:00,  1.75s/it]


In [ ]:
result_df.head(10)

,text,language,main_character
0,"The sun was setting, casting long shadows over...",English,John
1,"Le soleil se couchait, jetant de longues ombre...",French,Pierre
2,"El sol se estaba poniendo, proyectando largas ...",Spanish,Carlos
3,"La ciudad estaba llena de vida, sus calles lle...",Spanish,Juan
4,"La ville était pleine de vie, ses rues remplie...",French,Jean
5,"Die Stadt war voller Leben, ihre Straßen gefül...",German,Johann
6,Die Sonne ging unter und warf lange Schatten ü...,German,Hans
7,"В тихом уголке старого города, где узкие улочк...",Russian,Анна
8,In a small town nestled between the mountains ...,English,Laura
9,En un pequeño pueblo situado entre las montaña...,Spanish,Maria
